In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_data = pd.read_csv("/kaggle/input/titanic/train.csv")

train_data.head(5)

In [ ]:
# 데이터 구조 확인
train_data.info()

In [ ]:
train_data.describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.boxplot(x=train_data['Fare'])
plt.title("Boxplot - Fare")
plt.show()

Fare 기준으로 boxplot을 보았을 때 가장 오른쪽의 값이 이상치라고 생각이 들긴 하는데 가장 비싼 좌석의 데이터로 보임. 꼬리가 매우 긴 데이터를 보여 log 변환을 통해 정규분포에 가깝게 만든 후 IQR를 활용.

In [ ]:
import numpy as np

train_data['Fare_log'] = np.log1p(train_data['Fare']) # log(1+x)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.boxplot(x=train_data['Fare_log'])
plt.title("Boxplot - Fare_log")
plt.show()

In [ ]:
Q1 = train_data['Fare_log'].quantile(0.25)
Q3 = train_data['Fare_log'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outlier = train_data[(train_data['Fare_log'] < lower) | (train_data['Fare_log'] > upper)]
# train_clean = train_data[(train_data['Fare_log'] >= lower) & (train_data['Fare_log'] <= upper)]

In [ ]:
outlier[['Fare', 'Fare_log', 'Pclass', 'Survived']].sort_values('Fare', ascending=False).head(20)

In [ ]:
train_data.info()

In [ ]:
train_data['Age'] = train_data.groupby(['Sex', 'Pclass'])['Age'].transform(
    lambda x: x.fillna(x.median())
)

In [ ]:
train_data['Embarked'] = train_data['Embarked'].fillna(train_data['Embarked'].mode()[0])


In [ ]:
train_data.head(10)

In [ ]:
train_data['HasCabin'] = train_data['Cabin'].notnull().astype(int)

In [ ]:
clean_train = train_data.drop(columns=['Cabin'])

In [ ]:
clean_train['Sex'] = clean_train['Sex'].map({'male': 0, 'female': 1})
# clean_train = pd.get_dummies(clean_train, columns=['Embarked', 'Deck'], drop_first=True)
clean_train = pd.get_dummies(clean_train, columns=['Embarked'], drop_first=True)

In [ ]:
clean_train['FamilySize'] = clean_train['SibSp'] + clean_train['Parch'] + 1
clean_train.groupby('FamilySize')['Survived'].mean()

In [ ]:
clean_train['IsAlone'] = (clean_train['FamilySize'] == 1).astype(int)
clean_train.groupby('IsAlone')['Survived'].mean()

In [ ]:
clean_train['IsChild'] = (clean_train['Age'] <= 15).astype(int)

In [ ]:
clean_col_train = clean_train.drop(columns=["Name", "Ticket", "SibSp", "Parch"])

In [ ]:
clean_col_train.head(20)

In [ ]:
X = clean_col_train.drop(columns=['Survived', 'PassengerId'])
Y = clean_col_train['Survived']
train_passenger_ids = clean_col_train['PassengerId']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_valid, Y_train, Y_valid = train_test_split(X, Y, test_size = 0.2, random_state = 42, stratify=Y)

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1) 스케일링할 컬럼 선택
num_cols = ["Age", "Fare_log", "FamilySize"]

scaler = StandardScaler()

# 2) 훈련 데이터 스케일링 fit + transform
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# 3) 검증 데이터는 transform만 (fit 금지)
X_valid[num_cols] = scaler.transform(X_valid[num_cols])


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, Y_train)

In [ ]:
from sklearn.metrics import accuracy_score

# 예측
y_pred = model.predict(X_valid)

# 정확도
accuracy = accuracy_score(Y_valid, y_pred)
accuracy


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(Y_valid, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix

conf_mat = confusion_matrix(Y_valid, y_pred)
conf_mat


In [ ]:
X = clean_col_train.drop(columns=['Survived', 'PassengerId'])
Y = clean_col_train['Survived']
train_passenger_ids = clean_col_train['PassengerId']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_valid, Y_train, Y_valid = train_test_split(X, Y, test_size = 0.2, random_state = 42, stratify=Y)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=6,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)


rf.fit(X_train, Y_train)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = rf.predict(X_valid)

print("Accuracy:", accuracy_score(Y_valid, y_pred))
print(classification_report(Y_valid, y_pred))
print(confusion_matrix(Y_valid, y_pred))

In [ ]:
import pandas as pd
import numpy as np

importances = rf.feature_importances_
feature_names = X.columns

feat_imp = pd.DataFrame({'feature': feature_names, 'importance': importances})
feat_imp.sort_values('importance', ascending=False)

In [ ]:
clean_col_train = clean_train.drop(columns=["Name", "Ticket", "SibSp", "Parch"])

-----

In [ ]:
test = pd.read_csv('/kaggle/input/titanic/test.csv')

In [ ]:
test.info()

In [ ]:
test['Age'] = test.groupby(['Sex', 'Pclass'])['Age'].transform(
    lambda x: x.fillna(x.median())
)

In [ ]:
# Sex 인코딩
test['Sex'] = test['Sex'].map({'male': 0, 'female': 1})

# Fare 결측치
test['Fare'] = test['Fare'].fillna(test['Fare'].median())

# Fare_log
test['Fare_log'] = np.log1p(test['Fare'])

# HasCabin
test['HasCabin'] = test['Cabin'].notnull().astype(int)

# FamilySize
test['FamilySize'] = test['SibSp'] + test['Parch'] + 1

# IsAlone
test['IsAlone'] = (test['FamilySize'] == 1).astype(int)

# IsChild
test['IsChild'] = (test['Age'] < 15).astype(int)

# 불필요한 컬럼 제거
test = test.drop(columns=['Name', 'Ticket', 'Cabin', 'Deck'], errors='ignore')

# Embarked 더미 생성
test = pd.get_dummies(test, columns=['Embarked'], drop_first=True)

In [ ]:
for col in ['Embarked_Q', 'Embarked_S']:
    if col not in test.columns:
        test[col] = 0

In [ ]:
for col in X.columns:
    if col not in test.columns:
        test[col] = 0

X_test = test[X.columns]

In [ ]:
y_test_pred = rf.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': y_test_pred
})

submission.to_csv('submission.csv', index=False)